# Transformer From Scratch

Based on Neel Nanda's GPT-2 From Scratch

Adapted to use only HuggingFace and PyTorch, no other dependencies (Neel's EasyTransformers is great, but incompatible with some versions of TODO)
Training vs inference?

Why GPT-2?

TODO add extended desc

TODO put in git repo, organize, publish. see how they fit together

Hardware and CUDA

# Setup, Utilities, etc

In [ ]:
%pip install transformers
%pip install fancy_einsum
%pip install einops

In [ ]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
import torch
import torch.nn as nn
import numpy as np
import math
import tqdm.auto as tqdm

In [ ]:
#Config object to set params
@dataclass
class Config:
    d_model: int = 768 # size of residual string
    debug: bool = True
    layer_norm_epsilon: float = 1e-5 # added in layernorm to prevent division by 0
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12
    
cfg = Config()
print(cfg)

## Reference Model

Major adaptations:
- `model(x)` returns a `CausalLMOutputWithCrossAttentions`, which is a named tuple with `.logits`, `.past_key_values`, `.hidden_states`, `.attentions` - not just logits
- `run_with_cache` gives entries even for sub modules, use `c_attn` for concatenated QKV matrix `[B, T, 3*d_model]` before split
- Attention returns a tuple `(attn_output, present_key_value, attn_weights)`, so `output[0]` is the post-attention residual contribution that gets added back to the stream

In [ ]:
# We don't use Neel's EasyTransformers since dependencies have changed
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)
device = "cpu" # hard code, some operations not yet supported on mps

model_name = "gpt2"  # this is gpt2-small

reference_gpt2 = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

reference_gpt2.eval()

In [ ]:
print(reference_gpt2)

In [ ]:
# helpers

def run_with_cache(model, tokens):
    cache = {}
    hooks = []
    
    def make_hook(name):
        def hook(module, input, output):
            if isinstance(output, torch.Tensor):
                cache[name] = output.detach()
            elif isinstance(output, tuple) and isinstance(output[0], torch.Tensor):
                cache[name] = output[0].detach()
        return hook
    
    for name, module in model.named_modules():
        hooks.append(module.register_forward_hook(make_hook(name)))
        
    with torch.no_grad():
        out = model(tokens)
        
    for h in hooks:
        h.remove()
        
    return out.logits, cache

Key:
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 3072 (4 * d_model)
d_head = 64 (d_model / n_heads)

In [ ]:
# text to tokens
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = tokenizer.encode(reference_text, return_tensors="pt").to(device)
print(tokens)
print(tokens.shape) # [batch, position]
print(tokenizer.convert_ids_to_tokens(tokens[0].tolist()))

In [ ]:
# tokens to logits
logits, cache = run_with_cache(reference_gpt2, tokens)
print(logits.shape) # [batch, position]

In [ ]:
# logits to distribution
log_probs = logits.log_softmax(dim=-1)
probs = logits.softmax(dim=-1)
print(log_probs.shape)
print(probs.shape)

In [ ]:
# distribution to token
next_token = logits[0, -1].argmax(dim=-1) # -1 for the last token
print(next_token)
print(tokenizer.decode([next_token.item()]))

In [ ]:
# repeat!
next_tokens = torch.cat(
    [tokens, next_token.reshape(1,1)], dim=-1)

with torch.no_grad():
    new_output = reference_gpt2(next_tokens)
    
new_logits = new_output.logits # [batch, position, vocab_size]

print("New input (tokens):", next_tokens)
print(next_tokens.shape)
print("New input (text):", tokenizer.decode(next_tokens[0].tolist()))

print(new_logits.shape)

predicted_token_id = new_logits[0, -1].argmax(-1)
print("Predicted next token:", tokenizer.decode([predicted_token_id.item()]))

In [ ]:
# Activation Shapes
for activation_name, activation in cache.items():
    if not hasattr(activation, 'shape'):
        continue
    is_block_0 = "transformer.h.0" in activation_name
    is_top_level = "transformer.h" not in activation_name and activation_name != ""
    if is_block_0 or is_top_level: # only first block
        print(activation_name, tuple(activation.shape))

## Test

In [ ]:
# get_corner returns top-left corner of last two dimensions 
def get_corner(tensor, n=3):
    return tensor[..., :n, :n]

In [ ]:
# generate tensor of floats
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg)
    random_input = torch.randn(shape) # standard gaussian
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print(get_corner(output))
    return output

# generate tensor of ints
def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg)
    random_input = torch.randint(low=0, high=cfg.d_vocab, size=shape) # standard gaussian
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print(get_corner(output))
    return output

# testing our impl against base model
def load_gpt2_test(cls, gpt2_layer, input_name, gpt2_layer_output):
    cfg = Config(debug=True)
    layer = cls(cfg)
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    if isinstance(input_name, str):
        reference_input = cache[input_name]
    else:
        reference_input = input_name
    print("Input shape:", reference_input.shape)
    output = layer(reference_input) # output is our impl run on the input
    print("Output shape:", output.shape)
    print("Reference output shape:", gpt2_layer_output.shape)
    
    comparison = torch.isclose(output, gpt2_layer_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of values are correct")
    return output


# LayerNorm

- Applied at the start of each layer (MLP, attention)
- Normalizing function, taking an input vector and transforming it to have mean = 0 and variance = 1
- Also applies some non-linear scaling with the learned weights
- Neel describes this as an area where pretty cool math takes place - elementwise scaling and translation 

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(cfg.d_model)) # norm by default = 1
        self.bias = nn.Parameter(torch.zeros(cfg.d_model)) # mean by default = 0
        
    def forward(self, residual):
        # residual: [batch, position, d_model]
        if cfg.debug: print("Residual:", residual.shape)
        # einops.reduce(tensor, pattern, reduction, optional axes length) -> tensor
        residual = residual - einops.reduce(residual, "batch position d_model -> batch position 1", "mean")
        # calculate variance and square root, then add epsilon to ensure non-zero
        #   variance: sum of squares / n
        scale = (einops.reduce(residual.pow(2), "batch position d_model -> batch position 1", "mean") + cfg.layer_norm_epsilon).sqrt()
        residual = residual / scale
        normalized = residual * self.weight + self.bias
        if cfg.debug: print("Normalized:", residual.shape)
        return normalized

In [ ]:
x = rand_float_test(LayerNorm, [2, 4, 768])
reference_output = reference_gpt2.transformer.h[0].ln_1(x)
load_gpt2_test(LayerNorm, reference_gpt2.transformer.h[0].ln_1, x, reference_output)

# Embedding

- A lookup table from input token to vector
- Note that the process of tokenizing (raw input language to tokens) uses Byte-Pair Encoding (shoutout CS240E at Waterloo!). We start with only having tokens for individual letters (ie. `a=1, b=2, c=3` - a bit of an oversimplication), and merging the most common groups of sequences to create a vocabulary of short strings (ie. the pair `a` + `b` occurs often, so we create in our dictionary `ab=4`), some of which are complete words and others are not. This forms our vocabulary of tokens, where each string corresponds to an integer.
- We take an input `token_1, token_2, token_3, ...` and convert it into a tensor (matrix) via matrix multiplication with one-hot embedded vectors
- `d_vocab x d_model`

`token` (an integer) $\cdot \begin{bmatrix} 0 \\ 0 \\ 0 \\ ... \\ 1 \\ ... \\ 0 \end{bmatrix} $



In [ ]:
class Embed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.weight = nn.Parameter(torch.randn(cfg.d_vocab, cfg.d_model)) # why 0.02? why not?
        nn.init.normal_(self.weight, std=self.cfg.init_range)
    
    def forward(self, tokens):
        # tokens: [batch, position]
        if cfg.debug: print("Tokens:", tokens.shape)
        embed = self.weight[tokens, :] # [batch, position, d_model]
        if cfg.debug: print("Embeddings:", embed.shape)
        return embed

In [ ]:
x = torch.randint(low=0, high=cfg.d_vocab, size=[2,4])
reference_output = reference_gpt2.transformer.wte(x)
load_gpt2_test(Embed, reference_gpt2.transformer.wte, x, reference_output)

# Postitional Embedding
- We add information about the position of each token. 
- TODO check: Note that tokens that are close together should have a greater influence on one another - intuitively, in the sentence "the dancer leaped across the stage, and the lively crowd applauded", "lively" impacts the meaning of "crowd" a lot more than "dancer".

In [ ]:
class PosEmbed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.weight  = nn.Parameter(torch.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.weight, std=self.cfg.init_range)
        
    def forward(self, tokens):
        # tokens: [batch, position]
        if cfg.debug: print("Tokens:", tokens.shape)
        pos_embed = self.weight[:tokens.size(1), :] # [position, d_model]
        pos_embed = einops.repeat(pos_embed, "position d_model -> batch position d_model", batch=tokens.size(0))
        if cfg.debug: print("Positional Embeddings:", pos_embed.shape)
        return pos_embed
        

In [ ]:
x = torch.randint(low=0, high=cfg.n_ctx, size=[2,4])
reference_output = reference_gpt2.transformer.wpe(x)
load_gpt2_test(PosEmbed, reference_gpt2.transformer.wpe, x, reference_output)

# Attention
- This is the only step that moves information from a prior position in the sequence to the current one
- We parallelize this - we do this for each token for prefix token strings
- n heads etc

* Step 1: Produce attention pattern. For each destination token, produce a probability distribution over all previous tokens including itself
    * Linear map from input -> query, key shape [batch, position, head_index, d_head]
    * Dot product every pair of queries and keys to get attn_scores [batch, head_index, query_pos, key_pos] (query = dest, key = source)
    * Scale and mask attn_scores to make it lower triangular (remember when they said linear algebra would be useful?)
    * Softmax row-wise to get prob distribution along each key_pos dimension as attention pattern
* Step 2: Move info from source token to destination token using attention pattern (move via applying linear map)
    * Linear map from input -> value [batch, key_pos, head_index, d_head]
    * Mix along key_pos with attention pattern to get intermediate z [batch, query_pos, head_index, d_head]
    * Map to output [batch, position, d_model] (where position = query_pos summed over all heads)

In [ ]:
# do this yourself! recall - self heads vs others. diagonal patterns.
%pip install pysvelte
import pysvelte
pysvelte.AttentionMulti(tokens=reference_gpt2.to_str_tokens(reference_text), attention=cache['blocks.0.attn.hook_attn'].permute(1, 2, 0))

In [ ]:
class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range) # _ suffix means in-place method
        self.b_Q = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_K = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        self.b_K = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
        self.W_V = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head))) # why the double braces?
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        self.b_V = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
        
        self.W_O = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.b_O = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
        
        self.register_buffer("IGNORE", torch.tensor(-1e5, dtype=torch.float32, device="cuda"))
        
    def forward(self, normalized_resid_pre): # input is normalized residual
        # normalized_resid_pre: [batch, position, d_model]
        if cfg.debug: print("Normalized Residual (Pre):", normalized_resid_pre.shape)
        
        # query, key
        # note einsums are linear maps
        q = einsum("batch position d_model, n_heads d_model d_head -> batch position n_heads d_head", normalized_resid_pre, self.W_Q) + self.b_Q
        k = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_K) + self.b_K
        
        # attn scores are dot products of each pair of query, key
        # attn_scores are q by k transpose - model learns a "low-rank factorized matrix for each head", head learns d_model by d_model tensors?
        # W_O and W_V act on d_model dim, attention acts on position dimension - can permute order of applying attention
        attn_scores = einsum("batch query_pos n_heads d_head, batch key_pos n_heads d_head -> batch n_heads query_pos key_pos", q, k)
        attn_scores = attn_scores / math.sqrt(self.cfg.d_head)
        attn_scores = self.apply_causal_mask(attn_scores) # mask future positions?
        
        attn = attn_scores.softmax(dim=-1) # [batch, n_head, query_pos, key_pos]
        
        # value is the info we're moving from source position (query) to destination position (value)
        v = einsum("batch position d_model, n_heads d_model d_head -> batch position n_heads d_head", normalized_resid_pre, self.W_Q) + self.b_Q
        
        z = einsum("batch n_heads query_pos key_pos, batch key_pos n_heads d_head -> batch query_pos n_heads d_head", attn, v)
        
        attn_out = einsum("batch query_pos n_heads d_head, n_heads d_head d_model -> batch query_pos d_model", z, self.W_O) + self.b_O
        return attn_out
        
    def apply_causal_mask(self, attn_scores): # we mask all positions above the diagonal
        # attn_scores: [batch, n_heads, query_pos, key_pos]
        mask = torch.triu(torch.ones(attn_scores.size(-2), attn_scores.size(-1), device=attn_scores.device), diagonal=1).bool()
        attn_scores.masked_fill_(mask, self.IGNORE) # mask with (infinitely) negative values?
        return attn_scores

In [ ]:
# TODO adapt
rand_float_test(Attention, [2, 4, 768])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["blocks.0.ln1.hook_normalized"])

# MLP

* Linear map, GELU, linear map
* All of the non-linear transformations live here

In [ ]:
class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(torch.empty((cfg.d_model, cfg.d_mlp)))
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        self.b_in = nn.Parameter(torch.zeros((cfg.d_mlp)))
        self.W_out = nn.Parameter(torch.empty((cfg.d_mlp, cfg.d_model)))
        nn.init.normal_(self.W_out, std=self.cfg.init_range)
        self.b_out = nn.Parameter(torch.zeros((cfg.d_model)))
        
    def forward(self, normalized_resid_mid):
        # normalized_resid_mid: [batch, position, d_model]
        if cfg.debug: print("Normalized Residual (Mid):", normalized_resid_mid.shape)
        # note pre is pre-activation, then GELU to get post-activation
        pre = einsum("batch position d_model, d_model d_mlp -> batch position d_mlp", normalized_resid_mid, self.W_in) + self.b_in
        post = gelu_new(pre)
        mlp_out = einsum("batch position d_mlp, d_mlp d_model -> batch position d_model", post, self.W_out) + self.b_out
        return mlp_out

In [ ]:
# tests here

# Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)
        
    def forward(self, resid_pre):
        # resid_pre [batch, position, d_model]
        normalized_resid_pre = self.ln1(resid_pre)
        attn_out = self.attn(normalized_resid_pre)
        resid_mid = resid_pre + attn_out
        normalized_resid_mid = self.ln2(resid_mid)
        mlp_out = self.ml(normalized_resid_mid)
        resid_post = resid_mid + mlp_out
        return resid_post # go stare at the diagram!!! and then adapt
    
# tests

# Unembedding

- Transforms the final vector into a tensor of logits using a softmax function
$ softmax(x_i) = \frac{e^{x_i}}{\sum e^{x_j}} $
- This output is a tensor of logits, where we have one vector of size $d_{vocab}$ for each input token. 

$$
\begin{bmatrix}
a_0 & b_0 & ...\\
a_1 & b_1 & ...\\
... \\
a_{d} & b_d & ...
\end{bmatrix}
$$

In [ ]:
class Unembed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(torch.empty((cfg.d_model, cfg.d_vocab)))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(torch.zeros((cfg.d_vocab))) # bias - not used in GPT 2, but if folding in layernorm, creates bias
        # add a requires_grad=False flag
    
    def forward(self, normalized_resid_final):
        # normalized_resid_final [batch, position, d_model]
        if cfg.debug: print("Normalized Residual (Final):", normalized_resid_final.shape)
        logits = einsum("batch position d_model, d_model d_vocab -> batch position d_vocab", normalized_resid_final, self.W_U) + self.b_U
        return logits

# Full Transformer

In [ ]:
class DemoTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)
        
    def forward(self, tokens):
        # tokens [batch, position]
        embed = self.embed(tokens)
        pos_embed = self.pos_embed(tokens)
        residual = embed + pos_embed
        for block in self.blocks:
            residual = block(residual)
        normalized_resid_final = self.ln_final(residual)
        logits = self.unembed(normalized_resid_final)
        # logits [batch, position, logits]
        return logits
        

In [ ]:
rand_int_test(DemoTransformer, [2, 4])

# Using the Transformer

In [ ]:
demo_gpt2 = DemoTransformer(Config(debug=False))
demo_gpt2.load_state_dict(reference_gpt2.state_dict(), strict=False)

In [ ]:
test_string = """

"""

In [ ]:
test_tokens = tokenize(test_string)
demo_logits = demo_gpt2(test_tokens)